# Queue: 30k word-level DictaGen sets (plan parts 1 + 2 + 3), 4 arms x 3 seeds

Runs the 12 training notebooks below one after another: first seed 42 of all four arms, then seed 43, then seed 44.

| arm | training set (public HF) | lines |
|---|---|---|
| cursive fonts 30k | `cyttic/heb-llmgen30k-uni10aug-hhdgeom` | 30,000 |
| DiffusionPen augmented 30k (verified) | `cyttic/heb-llmgen30k-diffpenaug-hhdgeom` | 29,997 |
| joined 30k = DiffusionPen (verified) on parts 1 + 3 + cursive fonts on part 2 | `cyttic/heb-llmgen30k-join-diffpenaug-uni10aug-hhdgeom` | 29,997 |
| joined 30k, no-verify = the same, DiffusionPen parts NOT verified | `cyttic/heb-llmgen30k-join-diffpenaug-noverify-uni10aug-hhdgeom` | 30,000 |

Same 30k texts in all four arms; same recipe as the 10k / 20k runs (4 fixed epochs, so three times the steps of a 10k run).
Each run gets its own folder `~/work/queue_30k/<name>/` and pushes its model + results to its own public HF repo. A run whose HF repo already
holds `results.json` is skipped, so **re-running this notebook resumes** the queue. A failed run is reported and the queue goes on.

Start on the server: `./run.sh diffpen_words/queue_30k.ipynb` (inside tmux).

In [ ]:
import os, json, time, traceback
import papermill as pm
from huggingface_hub import HfApi
NBS = [
 "exp39_uni10aug30k_hhdgeom_llmgen_colab.ipynb",
 "exp39_diffpenaug30k_hhdgeom_llmgen_colab.ipynb",
 "exp39_join30k_diffpenaug_uni10aug_hhdgeom_llmgen_colab.ipynb",
 "exp39_joinnv30k_diffpenaugnv_uni10aug_hhdgeom_llmgen_colab.ipynb",
 "exp39_uni10aug30k_hhdgeom_llmgen_seed43_colab.ipynb",
 "exp39_diffpenaug30k_hhdgeom_llmgen_seed43_colab.ipynb",
 "exp39_join30k_diffpenaug_uni10aug_hhdgeom_llmgen_seed43_colab.ipynb",
 "exp39_joinnv30k_diffpenaugnv_uni10aug_hhdgeom_llmgen_seed43_colab.ipynb",
 "exp39_uni10aug30k_hhdgeom_llmgen_seed44_colab.ipynb",
 "exp39_diffpenaug30k_hhdgeom_llmgen_seed44_colab.ipynb",
 "exp39_join30k_diffpenaug_uni10aug_hhdgeom_llmgen_seed44_colab.ipynb",
 "exp39_joinnv30k_diffpenaugnv_uni10aug_hhdgeom_llmgen_seed44_colab.ipynb"
]
HERE = os.getcwd(); BASE = os.path.expanduser('~/work/queue_30k'); os.makedirs(BASE, exist_ok=True)
api = HfApi()

def out_repo(nb):
    src = ''.join(''.join(c['source']) for c in json.load(open(os.path.join(HERE, nb)))['cells'])
    return src.split('OUT_REPO = "', 1)[1].split('"', 1)[0]

def done(repo):
    try: return 'results.json' in api.list_repo_files(repo)
    except Exception: return False

status = {}
for i, nb in enumerate(NBS, 1):
    repo = out_repo(nb); name = nb.replace('_colab.ipynb', '')
    if done(repo): print(f'[{i}/{len(NBS)}] {name}: already on HF, skipped', flush=True); status[name] = 'skipped (done)'; continue
    wd = os.path.join(BASE, name); os.makedirs(wd, exist_ok=True); t0 = time.time()
    print(f'[{i}/{len(NBS)}] {name} -> {repo}  started {time.strftime("%H:%M")}', flush=True)
    try:
        pm.execute_notebook(os.path.join(HERE, nb), os.path.join(wd, nb), cwd=wd, log_output=True, kernel_name='python3')
        status[name] = f'ok ({(time.time() - t0) / 3600:.2f} h)' if done(repo) else 'finished, but results.json not on HF'
    except Exception as e:
        status[name] = 'FAILED: ' + repr(e)[:300]; traceback.print_exc()
    print(f'[{i}/{len(NBS)}] {name}: {status[name]}', flush=True)
print('\n==== queue summary'); [print(f'{k:60s} {v}') for k, v in status.items()]

## Results on HF (greedy and beam 4 with the notebooks' own LM; the LM1 numbers for the paper are computed afterwards with eval_diffpen10k_vs_font10k.py)

In [ ]:
from huggingface_hub import hf_hub_download
for nb in NBS:
    repo = out_repo(nb)
    if not done(repo): print(f'{repo}: no results'); continue
    T = json.load(open(hf_hub_download(repo, 'results.json', force_download=True)))['final']['test_decoding']
    print(repo, {k.split(' (')[0]: round(100 * v['cer'], 2) for k, v in T.items()})